In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [5]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [6]:
print("Total rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())

Total rows: 51071
Duplicate rows: 0


In [7]:
df.isnull().sum()

,0
text,0
emotion,0


In [8]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emotion in unique_emotions:
    emotion_numbers[emotion] = i
    i += 1
df['emotion'] = df['emotion'].map(emotion_numbers)

In [9]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [10]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [11]:
import string

def remove_punc(text):
    for punc in string.punctuation:
        text = text.replace(punc, '')
    return text

In [12]:
df['text'] = df['text'].apply(remove_punc)

In [13]:
def remove_numbers(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['text'] = df['text'].apply(remove_numbers)

In [14]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new += i
    return new

df['text'] = df['text'].apply(remove_emojis)

In [15]:
import nltk

In [16]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [17]:
# Downloading all required nltk resources together (was previously split
# across two cells, with punkt_tab only fetched right before it was needed —
# consolidated here so all downloads happen up front).
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [18]:
stop_words = set(stopwords.words('english'))

In [19]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [20]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if i not in stop_words:
            cleaned.append(i)
    return " ".join(cleaned)

In [21]:
df['text'] = df['text'].apply(remove)

In [22]:
df

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1
...,...,...
51066,feel fists clench right airline lost luggage,1
51067,feel wave hot fury rise took parking spot purpose,1
51068,feel endlessly infatuated partner life every t...,2
51069,feel paralyzed terror finding front door wide ...,4


In [23]:
print("Total:", len(df))
print(df["emotion"].value_counts())
print("Duplicate text:", df["text"].duplicated().sum())
print("Duplicate complete rows:", df.duplicated().sum())

Total: 51071
emotion
1    11506
4    10732
2     8400
0     6865
3     6818
5     6750
Name: count, dtype: int64
Duplicate text: 2984
Duplicate complete rows: 2977


In [24]:
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print("Rows after exact-duplicate removal:", len(df))
print("Unique texts:", df['text'].nunique())

Rows after exact-duplicate removal: 48087
Unique texts: 48087


In [25]:
from collections import defaultdict

NEAR_DUP_THRESHOLD = 0.75

df['_word_set'] = df['text'].apply(lambda t: frozenset(t.split()))
df['_word_count'] = df['_word_set'].apply(len)

buckets = defaultdict(list)
for idx, emotion, wc in zip(df.index, df['emotion'], df['_word_count']):
    buckets[(emotion, wc)].append(idx)

to_drop = set()
word_sets = df['_word_set']

for key, idxs in buckets.items():
    n = len(idxs)
    for i in range(n):
        idx_i = idxs[i]
        if idx_i in to_drop:
            continue
        set_i = word_sets[idx_i]
        for j in range(i + 1, n):
            idx_j = idxs[j]
            if idx_j in to_drop:
                continue
            set_j = word_sets[idx_j]
            inter = len(set_i & set_j)
            union = len(set_i) + len(set_j) - inter
            if union > 0 and inter / union >= NEAR_DUP_THRESHOLD:
                to_drop.add(idx_j)

print(f"Near-duplicate rows flagged: {len(to_drop)}")
df = df.drop(index=to_drop).drop(columns=['_word_set', '_word_count']).reset_index(drop=True)
print("Rows after near-duplicate removal:", len(df))
print(df['emotion'].value_counts())

Near-duplicate rows flagged: 6433
Rows after near-duplicate removal: 41654
emotion
1    9101
4    7851
5    6589
0    6512
2    6142
3    5459
Name: count, dtype: int64


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['emotion'],
    test_size=0.20,
    random_state=42,
    stratify=df['emotion']
)

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [29]:
bow_vectorizer = CountVectorizer()

In [30]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [31]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [32]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

MultinomialNB()

In [33]:
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))
print(confusion_matrix(y_test, pred_bow))
print(classification_report(y_test, pred_bow))

0.8844076341375585
[[1208   15    6    2   11   61]
 [ 146 1541    9    0   12  112]
 [  45    9 1051    0    1  122]
 [  33    1    4 1000   11   43]
 [ 101   14    9    7 1347   92]
 [  42   13   29    5    8 1221]]
              precision    recall  f1-score   support

           0       0.77      0.93      0.84      1303
           1       0.97      0.85      0.90      1820
           2       0.95      0.86      0.90      1228
           3       0.99      0.92      0.95      1092
           4       0.97      0.86      0.91      1570
           5       0.74      0.93      0.82      1318

    accuracy                           0.88      8331
   macro avg       0.90      0.89      0.89      8331
weighted avg       0.90      0.88      0.89      8331



In [34]:
tfid_vectorizer = TfidfVectorizer()
X_train_tfid = tfid_vectorizer.fit_transform(X_train)
X_test_tfid = tfid_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfid, y_train)

MultinomialNB()

In [35]:
pred_tfid = nb2_model.predict(X_test_tfid)
print(accuracy_score(y_test, pred_tfid))
print(confusion_matrix(y_test, pred_tfid))
print(classification_report(y_test, pred_tfid))

0.890169247389269
[[1197   20    5    2   15   64]
 [ 110 1589    7    0   19   95]
 [  44   14 1044    1    3  122]
 [  29    2    3  998   16   44]
 [  79   21    7    7 1368   88]
 [  38   21   25    3   11 1220]]
              precision    recall  f1-score   support

           0       0.80      0.92      0.85      1303
           1       0.95      0.87      0.91      1820
           2       0.96      0.85      0.90      1228
           3       0.99      0.91      0.95      1092
           4       0.96      0.87      0.91      1570
           5       0.75      0.93      0.83      1318

    accuracy                           0.89      8331
   macro avg       0.90      0.89      0.89      8331
weighted avg       0.90      0.89      0.89      8331



In [36]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(max_iter=1000)

In [37]:
logistic_model.fit(X_train_bow, y_train)

LogisticRegression(max_iter=1000)

In [38]:
log_pred = logistic_model.predict(X_test_bow)
print(accuracy_score(y_test, log_pred))
print(confusion_matrix(y_test, log_pred))
print(classification_report(y_test, log_pred))

0.9543872284239587
[[1234   22    7    2   19   19]
 [  30 1742    7    2   17   22]
 [   9    5 1159    0    3   52]
 [   5    1    1 1050   27    8]
 [  16   18    3    6 1516   11]
 [  20    9   27    7    5 1250]]
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1303
           1       0.97      0.96      0.96      1820
           2       0.96      0.94      0.95      1228
           3       0.98      0.96      0.97      1092
           4       0.96      0.97      0.96      1570
           5       0.92      0.95      0.93      1318

    accuracy                           0.95      8331
   macro avg       0.95      0.95      0.95      8331
weighted avg       0.95      0.95      0.95      8331



In [39]:
import joblib
joblib.dump(bow_vectorizer, "bow_vectorizer.pkl")
joblib.dump(logistic_model, "emotion_model.pkl")
joblib.dump(emotion_numbers, "emotion_numbers.pkl")

['emotion_numbers.pkl']

In [40]:
print(emotion_numbers)
print(df['emotion'].value_counts())
print(confusion_matrix(y_test, log_pred))

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}
emotion
1    9101
4    7851
5    6589
0    6512
2    6142
3    5459
Name: count, dtype: int64
[[1234   22    7    2   19   19]
 [  30 1742    7    2   17   22]
 [   9    5 1159    0    3   52]
 [   5    1    1 1050   27    8]
 [  16   18    3    6 1516   11]
 [  20    9   27    7    5 1250]]
